# Пространственные модели

In [1]:
import math
import collections

import pandas as pd
import numpy as np

from scipy import sparse
from scipy.linalg import qr
from scipy.optimize import minimize_scalar
from scipy.sparse.linalg import splu, spsolve
from scipy.stats import chi2

from sklearn.neighbors import NearestNeighbors

In [2]:
# грузим данные
dataset = pd.read_excel('../Dataset/objects_w_geo_prepared_V0_2j.xlsx')

# оставляем объявления внутри МКАД
dataset = dataset[dataset['inMKAD'] == 1].reset_index(drop=True)
dataset = dataset[dataset['AO'] != 'НЕОПР'].reset_index(drop=True)

# присваиваем зданиям номера
dataset['coords_str'] = [
    str(round(dataset.loc[i, 'avg_latitude'], 6)) + '_' + str(round(dataset.loc[i, 'avg_longtitude'], 6)) for i in dataset.index
]

coords = sorted(dataset['coords_str'].unique())
coords_to_num = {coords[i]: i for i in range(len(coords))}
dataset['address_index'] = [coords_to_num[x] for x in dataset['coords_str']]

print('Объявлений:', len(dataset))
print('Зданий:', dataset['address_index'].nunique())

Объявлений: 24332
Зданий: 3080


In [3]:
# признаки окружения
env_features = [
    'cafe_500', 'cafe_1000', 'cafe_2000',
    'fast_food_500', 'fast_food_1000', 'fast_food_2000',
    'restaurant_500', 'restaurant_1000', 'restaurant_2000',
    'bank_500', 'bank_1000', 'bank_2000',
    'supermarket_500', 'supermarket_1000', 'supermarket_2000',
    'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000',
    'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000',
    'histplaces_500', 'histplaces_1000', 'histplaces_2000'
]

office_classes = ['A', 'A+', 'B', 'B+', 'C']
ao_values = ['CВАО', 'ВАО', 'ЗАО', 'САО', 'СЗАО', 'ЮАО', 'ЮВАО', 'ЮЗАО']

all_rows = dataset.to_dict('records')

In [4]:
# агрегируем объявления до уровня здания
def first_mode(values):
    return collections.Counter(values).most_common(1)[0][0]


def make_building_dataset(rows, classes=None):
    if classes is not None:
        rows = [row for row in rows if row['office_class'] in classes]

    groups = collections.defaultdict(list)
    for row in rows:
        groups[row['address_index']].append(row)

    result = []

    for address_index, group in groups.items():
        row = {
            'address_index': address_index,
            'n_listings': len(group),
            'avg_latitude': np.mean([x['avg_latitude'] for x in group]),
            'avg_longtitude': np.mean([x['avg_longtitude'] for x in group]),
            'target': np.median([x['target'] for x in group]),
            'price_per_m2': np.median([x['price_per_m2'] for x in group]),
            'nl_floor': np.median([math.log1p(max(0, x['floor'])) for x in group]),
            'nl_maxfloor': np.median([math.log1p(max(0, x['maxfloor'])) for x in group]),
            'nl_space_m2': np.median([math.log1p(max(0, x['space_m2'])) for x in group]),
            'isCU': np.mean([x['isCU'] for x in group]),
            'isNonCU': np.mean([x['isNonCU'] for x in group]),
            'isEE': np.mean([x['isEE'] for x in group]),
            'isOfficeCenter': np.mean([x['isOfficeCenter'] for x in group]),
            'isMall': np.mean([x['isMall'] for x in group]),
            'inTTK': group[0]['inTTK'],
            'inGardenRing': group[0]['inGardenRing'],
            'AO': first_mode([x['AO'] for x in group]),
            'nl_minDistSubway': math.log1p(max(0, group[0]['minDistSubway'])),
            'nl_distanceToCenter': math.log1p(max(0, group[0]['distanceToCenter']))
        }

        for office_class in office_classes:
            row['office_class_' + office_class] = sum(
                x['office_class'] == office_class for x in group
            ) / len(group)

        for feature in env_features:
            row['nl_' + feature] = np.median([
                math.log1p(max(0, x[feature])) for x in group
            ])

        result.append(row)

    return pd.DataFrame(result)

In [5]:
def make_xy(buildings, add_class=True):
    buildings = buildings.copy()
    features = []

    if add_class:
        features += ['office_class_' + x for x in office_classes]

    features += [
        'nl_floor', 'nl_maxfloor',
        'isCU', 'isNonCU', 'isEE',
        'isOfficeCenter', 'isMall',
        'inTTK', 'inGardenRing'
    ]

    # ЦАО оставляем базовой категорией
    for ao in ao_values:
        buildings['AO_' + ao] = (buildings['AO'] == ao).astype(int)
        features.append('AO_' + ao)

    features += ['nl_space_m2', 'nl_minDistSubway']
    features += ['nl_' + x for x in env_features]
    features += ['nl_distanceToCenter']

    X = buildings[features].astype(float).copy()
    X.insert(0, 'const', 1.0)

    y = buildings['target'].to_numpy(dtype=float)
    coords = buildings[['avg_latitude', 'avg_longtitude']].to_numpy(dtype=float)

    return y, X.to_numpy(dtype=float), list(X.columns), coords


def drop_collinear(X, names):
    _, R, pivot = qr(X, mode='economic', pivoting=True)
    rank = np.sum(np.abs(np.diag(R)) > 1e-10 * np.abs(R[0, 0]))
    keep = sorted(pivot[:rank])
    return X[:, keep], [names[i] for i in keep]

In [6]:
# строим матрицу ближайших соседей
def make_weights(coords, n_neighbors=8):
    coords_rad = np.radians(coords)

    model = NearestNeighbors(
        n_neighbors=min(n_neighbors + 1, len(coords)),
        metric='haversine',
        algorithm='ball_tree'
    )
    model.fit(coords_rad)
    _, neighbors = model.kneighbors(coords_rad)

    rows = []
    cols = []

    for i in range(len(coords)):
        for j in neighbors[i, 1:]:
            rows.append(i)
            cols.append(int(j))

    W = sparse.csr_matrix(
        (np.ones(len(rows)), (rows, cols)),
        shape=(len(coords), len(coords))
    )

    W = ((W) > 0).astype(float).tocsr()
    row_sums = np.asarray(W.sum(axis=1)).ravel()
    W = sparse.diags(1 / row_sums) @ W

    return W.tocsr()

In [7]:
# обычная линейная модель
def fit_ols(y, X):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    fitted = X @ beta
    residuals = y - fitted

    n = len(y)
    k = X.shape[1]
    sse = float(residuals @ residuals)
    sigma2 = sse / n

    log_likelihood = -0.5 * n * (np.log(2 * np.pi) + 1 + np.log(sigma2))
    r2 = 1 - sse / float((y - y.mean()) @ (y - y.mean()))
    adjusted_r2 = 1 - (1 - r2) * (n - 1) / (n - k)
    aic = -2 * log_likelihood + 2 * (k + 1)

    return beta, residuals, log_likelihood, r2, adjusted_r2, aic


# индекс Морана
def moran_test(values, W, n_permutations=499):
    values = np.asarray(values, dtype=float)
    values = values - values.mean()

    n = len(values)
    s0 = float(W.sum())
    moran = float(n / s0 * (values @ (W @ values)) / (values @ values))

    rng = np.random.default_rng(123)
    permutations = []

    for i in range(n_permutations):
        permuted = rng.permutation(values)
        value = n / s0 * (permuted @ (W @ permuted)) / (permuted @ permuted)
        permutations.append(value)

    p_value = (1 + np.sum(np.asarray(permutations) >= moran)) / (n_permutations + 1)
    return moran, p_value

In [8]:
def sparse_logdet(A):
    lu = splu(A.tocsc())
    return float(np.sum(np.log(np.abs(lu.U.diagonal()))))


# spatial error model
def fit_sem(y, X, W):
    n = len(y)
    I = sparse.eye(n, format='csr')

    def objective(lambda_value, return_model=False):
        A = I - lambda_value * W
        y_new = A @ y
        X_new = A @ X

        beta = np.linalg.lstsq(X_new, y_new, rcond=None)[0]
        errors = y_new - X_new @ beta

        sse = float(errors @ errors)
        sigma2 = sse / n
        log_likelihood = sparse_logdet(A) - 0.5 * n * (
            np.log(2 * np.pi) + 1 + np.log(sigma2)
        )

        if return_model:
            return log_likelihood, beta, errors
        return -log_likelihood

    result = minimize_scalar(
        objective,
        bounds=(-0.95, 0.95),
        method='bounded',
        options={'xatol': 1e-6, 'maxiter': 100}
    )

    lambda_value = float(result.x)
    log_likelihood, beta, errors = objective(lambda_value, True)
    residuals = y - X @ beta
    r2 = 1 - float(residuals @ residuals) / float((y - y.mean()) @ (y - y.mean()))
    aic = -2 * log_likelihood + 2 * (X.shape[1] + 2)

    return lambda_value, beta, errors, residuals, log_likelihood, r2, aic


# spatial lag model
def fit_sar(y, X, W):
    n = len(y)
    I = sparse.eye(n, format='csr')
    Wy = W @ y
    xtx_inverse = np.linalg.pinv(X.T @ X)

    def objective(rho_value, return_model=False):
        A = I - rho_value * W
        y_new = y - rho_value * Wy

        beta = xtx_inverse @ (X.T @ y_new)
        errors = y_new - X @ beta

        sse = float(errors @ errors)
        sigma2 = sse / n
        log_likelihood = sparse_logdet(A) - 0.5 * n * (
            np.log(2 * np.pi) + 1 + np.log(sigma2)
        )

        if return_model:
            return log_likelihood, beta, errors, A
        return -log_likelihood

    result = minimize_scalar(
        objective,
        bounds=(-0.95, 0.95),
        method='bounded',
        options={'xatol': 1e-6, 'maxiter': 100}
    )

    rho_value = float(result.x)
    log_likelihood, beta, errors, A = objective(rho_value, True)
    fitted = spsolve(A, X @ beta)
    residuals = y - fitted
    r2 = 1 - float(residuals @ residuals) / float((y - y.mean()) @ (y - y.mean()))
    aic = -2 * log_likelihood + 2 * (X.shape[1] + 2)

    return rho_value, beta, errors, residuals, log_likelihood, r2, aic

In [9]:
key_features = [
    'nl_minDistSubway',
    'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000',
    'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
    'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
    'nl_distanceToCenter', 'nl_space_m2'
]


def key_coefficients(sample, names, ols_beta, sar_beta, sem_beta):
    rows = []

    for feature in key_features:
        rows.append({
            'sample': sample,
            'variable': feature,
            'OLS_coef': ols_beta[names.index(feature)] if feature in names else np.nan,
            'SAR_coef': sar_beta[names.index(feature)] if feature in names else np.nan,
            'SEM_coef': sem_beta[names.index(feature)] if feature in names else np.nan
        })

    return pd.DataFrame(rows)

In [10]:
# общая выборка
buildings_all = make_building_dataset(all_rows)
print('Зданий:', len(buildings_all), 'Объявлений:', int(buildings_all['n_listings'].sum()))

y_all, X_all, names_all, coords_all = make_xy(buildings_all, add_class=True)
X_all, names_all = drop_collinear(X_all, names_all)
W_all = make_weights(coords_all, 8)

ols_beta_all, ols_resid_all, ols_ll_all, ols_r2_all, ols_adj_r2_all, ols_aic_all = fit_ols(y_all, X_all)
ols_moran_all, ols_moran_p_all = moran_test(ols_resid_all, W_all)

sar_rho_all, sar_beta_all, sar_errors_all, sar_resid_all, sar_ll_all, sar_r2_all, sar_aic_all = fit_sar(y_all, X_all, W_all)
sar_moran_all, sar_moran_p_all = moran_test(sar_errors_all, W_all)

sem_lambda_all, sem_beta_all, sem_errors_all, sem_resid_all, sem_ll_all, sem_r2_all, sem_aic_all = fit_sem(y_all, X_all, W_all)
sem_moran_all, sem_moran_p_all = moran_test(sem_errors_all, W_all)

lr_stat_all = 2 * (sar_ll_all - ols_ll_all)
lr_p_all = chi2.sf(max(0, lr_stat_all), df=1)

result_all = {
    'sample': 'All',
    'n_buildings': len(buildings_all),
    'n_listings': int(buildings_all['n_listings'].sum()),
    'OLS_R2': ols_r2_all,
    'OLS_adj_R2': ols_adj_r2_all,
    'OLS_AIC': ols_aic_all,
    'OLS_Moran_I': ols_moran_all,
    'OLS_Moran_p': ols_moran_p_all,
    'SAR_rho': sar_rho_all,
    'SAR_R2': sar_r2_all,
    'SAR_AIC': sar_aic_all,
    'LR_test_p': lr_p_all,
    'SAR_error_Moran_I': sar_moran_all,
    'SAR_error_Moran_p': sar_moran_p_all,
    'SEM_lambda': sem_lambda_all,
    'SEM_R2': sem_r2_all,
    'SEM_AIC': sem_aic_all,
    'SEM_error_Moran_I': sem_moran_all,
    'SEM_error_Moran_p': sem_moran_p_all
}

coef_all = key_coefficients('All', names_all, ols_beta_all, sar_beta_all, sem_beta_all)

display(pd.DataFrame([result_all]))
display(coef_all)

Зданий: 3080 Объявлений: 24332


,sample,n_buildings,n_listings,OLS_R2,OLS_adj_R2,OLS_AIC,OLS_Moran_I,OLS_Moran_p,SAR_rho,SAR_R2,SAR_AIC,LR_test_p,SAR_error_Moran_I,SAR_error_Moran_p,SEM_lambda,SEM_R2,SEM_AIC,SEM_error_Moran_I,SEM_error_Moran_p
0,All,3080,24332,0.453717,0.443782,3162.321249,0.057306,0.002,0.202671,0.455916,3115.658586,3.039991e-12,-0.000629,0.482,0.215651,0.453161,3125.175107,-0.00052,0.484


,sample,variable,OLS_coef,SAR_coef,SEM_coef
0,All,nl_minDistSubway,-0.094294,-0.083137,-0.097541
1,All,nl_restaurant_500,0.080208,0.063522,0.074623
2,All,nl_restaurant_1000,0.020689,0.008947,0.020744
3,All,nl_restaurant_2000,0.095149,0.068582,0.084912
4,All,nl_cafe_500,0.029820,0.025357,0.026755
5,All,nl_cafe_1000,-0.025456,-0.019962,-0.018266
6,All,nl_cafe_2000,-0.012150,-0.007748,-0.003837
7,All,nl_supermarket_500,-0.000384,0.001569,-0.001488
8,All,nl_supermarket_1000,-0.028318,-0.020008,-0.025302
9,All,nl_supermarket_2000,-0.058819,-0.041939,-0.052015


In [11]:
# класс A
buildings_a = make_building_dataset(all_rows, ['A', 'A+'])
print('Зданий:', len(buildings_a), 'Объявлений:', int(buildings_a['n_listings'].sum()))

y_a, X_a, names_a, coords_a = make_xy(buildings_a, add_class=False)
X_a, names_a = drop_collinear(X_a, names_a)
W_a = make_weights(coords_a, 8)

ols_beta_a, ols_resid_a, ols_ll_a, ols_r2_a, ols_adj_r2_a, ols_aic_a = fit_ols(y_a, X_a)
ols_moran_a, ols_moran_p_a = moran_test(ols_resid_a, W_a)

sar_rho_a, sar_beta_a, sar_errors_a, sar_resid_a, sar_ll_a, sar_r2_a, sar_aic_a = fit_sar(y_a, X_a, W_a)
sar_moran_a, sar_moran_p_a = moran_test(sar_errors_a, W_a)

sem_lambda_a, sem_beta_a, sem_errors_a, sem_resid_a, sem_ll_a, sem_r2_a, sem_aic_a = fit_sem(y_a, X_a, W_a)
sem_moran_a, sem_moran_p_a = moran_test(sem_errors_a, W_a)

lr_stat_a = 2 * (sar_ll_a - ols_ll_a)
lr_p_a = chi2.sf(max(0, lr_stat_a), df=1)

result_a = {
    'sample': 'A',
    'n_buildings': len(buildings_a),
    'n_listings': int(buildings_a['n_listings'].sum()),
    'OLS_R2': ols_r2_a,
    'OLS_adj_R2': ols_adj_r2_a,
    'OLS_AIC': ols_aic_a,
    'OLS_Moran_I': ols_moran_a,
    'OLS_Moran_p': ols_moran_p_a,
    'SAR_rho': sar_rho_a,
    'SAR_R2': sar_r2_a,
    'SAR_AIC': sar_aic_a,
    'LR_test_p': lr_p_a,
    'SAR_error_Moran_I': sar_moran_a,
    'SAR_error_Moran_p': sar_moran_p_a,
    'SEM_lambda': sem_lambda_a,
    'SEM_R2': sem_r2_a,
    'SEM_AIC': sem_aic_a,
    'SEM_error_Moran_I': sem_moran_a,
    'SEM_error_Moran_p': sem_moran_p_a
}

coef_a = key_coefficients('A', names_a, ols_beta_a, sar_beta_a, sem_beta_a)

display(pd.DataFrame([result_a]))
display(coef_a)

Зданий: 498 Объявлений: 6330


,sample,n_buildings,n_listings,OLS_R2,OLS_adj_R2,OLS_AIC,OLS_Moran_I,OLS_Moran_p,SAR_rho,SAR_R2,SAR_AIC,LR_test_p,SAR_error_Moran_I,SAR_error_Moran_p,SEM_lambda,SEM_R2,SEM_AIC,SEM_error_Moran_I,SEM_error_Moran_p
0,A,498,6330,0.47167,0.412573,513.487225,0.005834,0.314,0.137643,0.474828,512.855435,0.104744,-0.014642,0.722,0.053652,0.471509,515.330225,0.000706,0.42


,sample,variable,OLS_coef,SAR_coef,SEM_coef
0,A,nl_minDistSubway,-0.085496,-0.079919,-0.084019
1,A,nl_restaurant_500,0.067459,0.061675,0.066041
2,A,nl_restaurant_1000,0.031392,0.032168,0.033641
3,A,nl_restaurant_2000,-0.098473,-0.097475,-0.097278
4,A,nl_cafe_500,0.046453,0.042856,0.045511
5,A,nl_cafe_1000,0.105510,0.106074,0.108350
6,A,nl_cafe_2000,0.264059,0.265025,0.269816
7,A,nl_supermarket_500,-0.014653,-0.012943,-0.014450
8,A,nl_supermarket_1000,-0.071550,-0.063694,-0.071372
9,A,nl_supermarket_2000,-0.203981,-0.176128,-0.196973


In [12]:
# класс B
buildings_b = make_building_dataset(all_rows, ['B', 'B+'])
print('Зданий:', len(buildings_b), 'Объявлений:', int(buildings_b['n_listings'].sum()))

y_b, X_b, names_b, coords_b = make_xy(buildings_b, add_class=False)
X_b, names_b = drop_collinear(X_b, names_b)
W_b = make_weights(coords_b, 8)

ols_beta_b, ols_resid_b, ols_ll_b, ols_r2_b, ols_adj_r2_b, ols_aic_b = fit_ols(y_b, X_b)
ols_moran_b, ols_moran_p_b = moran_test(ols_resid_b, W_b)

sar_rho_b, sar_beta_b, sar_errors_b, sar_resid_b, sar_ll_b, sar_r2_b, sar_aic_b = fit_sar(y_b, X_b, W_b)
sar_moran_b, sar_moran_p_b = moran_test(sar_errors_b, W_b)

sem_lambda_b, sem_beta_b, sem_errors_b, sem_resid_b, sem_ll_b, sem_r2_b, sem_aic_b = fit_sem(y_b, X_b, W_b)
sem_moran_b, sem_moran_p_b = moran_test(sem_errors_b, W_b)

lr_stat_b = 2 * (sar_ll_b - ols_ll_b)
lr_p_b = chi2.sf(max(0, lr_stat_b), df=1)

result_b = {
    'sample': 'B',
    'n_buildings': len(buildings_b),
    'n_listings': int(buildings_b['n_listings'].sum()),
    'OLS_R2': ols_r2_b,
    'OLS_adj_R2': ols_adj_r2_b,
    'OLS_AIC': ols_aic_b,
    'OLS_Moran_I': ols_moran_b,
    'OLS_Moran_p': ols_moran_p_b,
    'SAR_rho': sar_rho_b,
    'SAR_R2': sar_r2_b,
    'SAR_AIC': sar_aic_b,
    'LR_test_p': lr_p_b,
    'SAR_error_Moran_I': sar_moran_b,
    'SAR_error_Moran_p': sar_moran_p_b,
    'SEM_lambda': sem_lambda_b,
    'SEM_R2': sem_r2_b,
    'SEM_AIC': sem_aic_b,
    'SEM_error_Moran_I': sem_moran_b,
    'SEM_error_Moran_p': sem_moran_p_b
}

coef_b = key_coefficients('B', names_b, ols_beta_b, sar_beta_b, sem_beta_b)

display(pd.DataFrame([result_b]))
display(coef_b)

Зданий: 1690 Объявлений: 11707


,sample,n_buildings,n_listings,OLS_R2,OLS_adj_R2,OLS_AIC,OLS_Moran_I,OLS_Moran_p,SAR_rho,SAR_R2,SAR_AIC,LR_test_p,SAR_error_Moran_I,SAR_error_Moran_p,SEM_lambda,SEM_R2,SEM_AIC,SEM_error_Moran_I,SEM_error_Moran_p
0,B,1690,11707,0.40142,0.38316,1703.372441,0.047722,0.002,0.204483,0.403961,1682.955124,0.000002,-0.008334,0.768,0.196517,0.400516,1689.079873,-0.002256,0.546


,sample,variable,OLS_coef,SAR_coef,SEM_coef
0,B,nl_minDistSubway,-0.082467,-0.073858,-0.074931
1,B,nl_restaurant_500,0.077810,0.066296,0.076628
2,B,nl_restaurant_1000,0.039107,0.029530,0.042377
3,B,nl_restaurant_2000,0.196121,0.149735,0.168841
4,B,nl_cafe_500,0.056352,0.053168,0.053511
5,B,nl_cafe_1000,-0.046298,-0.047022,-0.046886
6,B,nl_cafe_2000,-0.012989,0.005173,0.006876
7,B,nl_supermarket_500,-0.004872,-0.001746,-0.003534
8,B,nl_supermarket_1000,-0.052373,-0.042593,-0.045278
9,B,nl_supermarket_2000,-0.127290,-0.103117,-0.118024


In [13]:
# класс C
buildings_c = make_building_dataset(all_rows, ['C'])
print('Зданий:', len(buildings_c), 'Объявлений:', int(buildings_c['n_listings'].sum()))

y_c, X_c, names_c, coords_c = make_xy(buildings_c, add_class=False)
X_c, names_c = drop_collinear(X_c, names_c)
W_c = make_weights(coords_c, 8)

ols_beta_c, ols_resid_c, ols_ll_c, ols_r2_c, ols_adj_r2_c, ols_aic_c = fit_ols(y_c, X_c)
ols_moran_c, ols_moran_p_c = moran_test(ols_resid_c, W_c)

sar_rho_c, sar_beta_c, sar_errors_c, sar_resid_c, sar_ll_c, sar_r2_c, sar_aic_c = fit_sar(y_c, X_c, W_c)
sar_moran_c, sar_moran_p_c = moran_test(sar_errors_c, W_c)

sem_lambda_c, sem_beta_c, sem_errors_c, sem_resid_c, sem_ll_c, sem_r2_c, sem_aic_c = fit_sem(y_c, X_c, W_c)
sem_moran_c, sem_moran_p_c = moran_test(sem_errors_c, W_c)

lr_stat_c = 2 * (sar_ll_c - ols_ll_c)
lr_p_c = chi2.sf(max(0, lr_stat_c), df=1)

result_c = {
    'sample': 'C',
    'n_buildings': len(buildings_c),
    'n_listings': int(buildings_c['n_listings'].sum()),
    'OLS_R2': ols_r2_c,
    'OLS_adj_R2': ols_adj_r2_c,
    'OLS_AIC': ols_aic_c,
    'OLS_Moran_I': ols_moran_c,
    'OLS_Moran_p': ols_moran_p_c,
    'SAR_rho': sar_rho_c,
    'SAR_R2': sar_r2_c,
    'SAR_AIC': sar_aic_c,
    'LR_test_p': lr_p_c,
    'SAR_error_Moran_I': sar_moran_c,
    'SAR_error_Moran_p': sar_moran_p_c,
    'SEM_lambda': sem_lambda_c,
    'SEM_R2': sem_r2_c,
    'SEM_AIC': sem_aic_c,
    'SEM_error_Moran_I': sem_moran_c,
    'SEM_error_Moran_p': sem_moran_p_c
}

coef_c = key_coefficients('C', names_c, ols_beta_c, sar_beta_c, sem_beta_c)

display(pd.DataFrame([result_c]))
display(coef_c)

Зданий: 229 Объявлений: 395


,sample,n_buildings,n_listings,OLS_R2,OLS_adj_R2,OLS_AIC,OLS_Moran_I,OLS_Moran_p,SAR_rho,SAR_R2,SAR_AIC,LR_test_p,SAR_error_Moran_I,SAR_error_Moran_p,SEM_lambda,SEM_R2,SEM_AIC,SEM_error_Moran_I,SEM_error_Moran_p
0,C,229,395,0.535279,0.404739,245.981853,-0.027608,0.756,0.038807,0.536466,247.901718,0.777114,-0.032611,0.808,-0.949999,0.490837,240.609842,-0.012178,0.554


,sample,variable,OLS_coef,SAR_coef,SEM_coef
0,C,nl_minDistSubway,-0.034315,-0.034287,-0.023861
1,C,nl_restaurant_500,0.168875,0.167846,0.204084
2,C,nl_restaurant_1000,-0.026444,-0.029083,0.030364
3,C,nl_restaurant_2000,0.218175,0.212744,0.272377
4,C,nl_cafe_500,0.055539,0.056747,-0.012431
5,C,nl_cafe_1000,-0.054805,-0.056816,-0.051571
6,C,nl_cafe_2000,0.005567,0.005403,-0.040924
7,C,nl_supermarket_500,0.026843,0.027350,-0.002383
8,C,nl_supermarket_1000,-0.047710,-0.043010,-0.173064
9,C,nl_supermarket_2000,-0.002495,0.001198,-0.135567


In [14]:
# сводные таблицы
model_comparison = pd.DataFrame([
    result_all,
    result_a,
    result_b,
    result_c
])

key_coefficients_table = pd.concat([
    coef_all,
    coef_a,
    coef_b,
    coef_c
], ignore_index=True)

model_comparison.to_csv('./spatial_result/spatial_model_comparison.csv', index=False, encoding='utf-8-sig')
key_coefficients_table.to_csv('./spatial_result/spatial_key_coefficients.csv', index=False, encoding='utf-8-sig')

display(model_comparison)
display(key_coefficients_table)

,sample,n_buildings,n_listings,OLS_R2,OLS_adj_R2,OLS_AIC,OLS_Moran_I,OLS_Moran_p,SAR_rho,SAR_R2,SAR_AIC,LR_test_p,SAR_error_Moran_I,SAR_error_Moran_p,SEM_lambda,SEM_R2,SEM_AIC,SEM_error_Moran_I,SEM_error_Moran_p
0,All,3080,24332,0.453717,0.443782,3162.321249,0.057306,0.002,0.202671,0.455916,3115.658586,3.039991e-12,-0.000629,0.482,0.215651,0.453161,3125.175107,-0.000520,0.484
1,A,498,6330,0.471670,0.412573,513.487225,0.005834,0.314,0.137643,0.474828,512.855435,1.047435e-01,-0.014642,0.722,0.053652,0.471509,515.330225,0.000706,0.420
2,B,1690,11707,0.401420,0.383160,1703.372441,0.047722,0.002,0.204483,0.403961,1682.955124,2.193870e-06,-0.008334,0.768,0.196517,0.400516,1689.079873,-0.002256,0.546
3,C,229,395,0.535279,0.404739,245.981853,-0.027608,0.756,0.038807,0.536466,247.901718,7.771144e-01,-0.032611,0.808,-0.949999,0.490837,240.609842,-0.012178,0.554


,sample,variable,OLS_coef,SAR_coef,SEM_coef
0,All,nl_minDistSubway,-0.094294,-0.083137,-0.097541
1,All,nl_restaurant_500,0.080208,0.063522,0.074623
2,All,nl_restaurant_1000,0.020689,0.008947,0.020744
3,All,nl_restaurant_2000,0.095149,0.068582,0.084912
4,All,nl_cafe_500,0.029820,0.025357,0.026755
5,All,nl_cafe_1000,-0.025456,-0.019962,-0.018266
6,All,nl_cafe_2000,-0.012150,-0.007748,-0.003837
7,All,nl_supermarket_500,-0.000384,0.001569,-0.001488
8,All,nl_supermarket_1000,-0.028318,-0.020008,-0.025302
9,All,nl_supermarket_2000,-0.058819,-0.041939,-0.052015


In [15]:
# проверяем устойчивость для общей выборки
W_all_5 = make_weights(coords_all, 5)
W_all_12 = make_weights(coords_all, 12)

moran_all_5, moran_all_p_5 = moran_test(ols_resid_all, W_all_5)
rho_all_5, _, errors_all_5, _, _, _, aic_all_5 = fit_sar(y_all, X_all, W_all_5)
moran_sar_all_5, moran_sar_all_p_5 = moran_test(errors_all_5, W_all_5)

moran_all_12, moran_all_p_12 = moran_test(ols_resid_all, W_all_12)
rho_all_12, _, errors_all_12, _, _, _, aic_all_12 = fit_sar(y_all, X_all, W_all_12)
moran_sar_all_12, moran_sar_all_p_12 = moran_test(errors_all_12, W_all_12)

robustness_all = pd.DataFrame([
    {'sample': 'All', 'n_neighbors': 5, 'OLS_Moran_I': moran_all_5, 'OLS_Moran_p': moran_all_p_5, 'SAR_rho': rho_all_5, 'SAR_AIC': aic_all_5, 'SAR_error_Moran_I': moran_sar_all_5, 'SAR_error_Moran_p': moran_sar_all_p_5},
    {'sample': 'All', 'n_neighbors': 8, 'OLS_Moran_I': ols_moran_all, 'OLS_Moran_p': ols_moran_p_all, 'SAR_rho': sar_rho_all, 'SAR_AIC': sar_aic_all, 'SAR_error_Moran_I': sar_moran_all, 'SAR_error_Moran_p': sar_moran_p_all},
    {'sample': 'All', 'n_neighbors': 12, 'OLS_Moran_I': moran_all_12, 'OLS_Moran_p': moran_all_p_12, 'SAR_rho': rho_all_12, 'SAR_AIC': aic_all_12, 'SAR_error_Moran_I': moran_sar_all_12, 'SAR_error_Moran_p': moran_sar_all_p_12}
])

display(robustness_all)

,sample,n_neighbors,OLS_Moran_I,OLS_Moran_p,SAR_rho,SAR_AIC,SAR_error_Moran_I,SAR_error_Moran_p
0,All,5,0.078784,0.002,0.193338,3101.728101,-0.002561,0.550
1,All,8,0.057306,0.002,0.202671,3115.658586,-0.000629,0.482
2,All,12,0.046669,0.002,0.219386,3122.037930,0.000870,0.426


In [16]:
# проверяем устойчивость для класса B
W_b_5 = make_weights(coords_b, 5)
W_b_12 = make_weights(coords_b, 12)

moran_b_5, moran_b_p_5 = moran_test(ols_resid_b, W_b_5)
rho_b_5, _, errors_b_5, _, _, _, aic_b_5 = fit_sar(y_b, X_b, W_b_5)
moran_sar_b_5, moran_sar_b_p_5 = moran_test(errors_b_5, W_b_5)

moran_b_12, moran_b_p_12 = moran_test(ols_resid_b, W_b_12)
rho_b_12, _, errors_b_12, _, _, _, aic_b_12 = fit_sar(y_b, X_b, W_b_12)
moran_sar_b_12, moran_sar_b_p_12 = moran_test(errors_b_12, W_b_12)

robustness_b = pd.DataFrame([
    {'sample': 'B', 'n_neighbors': 5, 'OLS_Moran_I': moran_b_5, 'OLS_Moran_p': moran_b_p_5, 'SAR_rho': rho_b_5, 'SAR_AIC': aic_b_5, 'SAR_error_Moran_I': moran_sar_b_5, 'SAR_error_Moran_p': moran_sar_b_p_5},
    {'sample': 'B', 'n_neighbors': 8, 'OLS_Moran_I': ols_moran_b, 'OLS_Moran_p': ols_moran_p_b, 'SAR_rho': sar_rho_b, 'SAR_AIC': sar_aic_b, 'SAR_error_Moran_I': sar_moran_b, 'SAR_error_Moran_p': sar_moran_p_b},
    {'sample': 'B', 'n_neighbors': 12, 'OLS_Moran_I': moran_b_12, 'OLS_Moran_p': moran_b_p_12, 'SAR_rho': rho_b_12, 'SAR_AIC': aic_b_12, 'SAR_error_Moran_I': moran_sar_b_12, 'SAR_error_Moran_p': moran_sar_b_p_12}
])

knn_robustness = pd.concat([robustness_all, robustness_b], ignore_index=True)
knn_robustness.to_csv('./spatial_result/spatial_knn_robustness.csv', index=False, encoding='utf-8-sig')

display(knn_robustness)

,sample,n_neighbors,OLS_Moran_I,OLS_Moran_p,SAR_rho,SAR_AIC,SAR_error_Moran_I,SAR_error_Moran_p
0,All,5,0.078784,0.002,0.193338,3101.728101,-0.002561,0.550
1,All,8,0.057306,0.002,0.202671,3115.658586,-0.000629,0.482
2,All,12,0.046669,0.002,0.219386,3122.037930,0.000870,0.426
3,B,5,0.066127,0.002,0.176059,1680.356037,-0.009097,0.748
4,B,8,0.047722,0.002,0.204483,1682.955124,-0.008334,0.768
5,B,12,0.026882,0.002,0.172912,1693.904876,-0.006839,0.754
